In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
import lightgbm as lgb



In [2]:
train = pd.read_csv("../input/ventilator-pressure-prediction/train.csv")
test = pd.read_csv("../input/ventilator-pressure-prediction/test.csv")




In [3]:
def rmspe(y_true, y_pred):
    """Root Mean Square Percentage Error (used only for custom evaluation)."""
    return np.sqrt(np.mean(np.square((y_true - y_pred) / y_true)))


def feval_rmspe(y_pred, lgb_train):
    y_true = lgb_train.get_label()
    return "RMSPE", rmspe(y_true, y_pred), False


def train_and_evaluate_lgb(train_df, test_df, params):
    # Features: all columns except id and target pressure
    features = [c for c in train_df.columns if c not in {"id", "pressure"}]
    y = train_df["pressure"]

    oof_preds = np.zeros(train_df.shape[0])
    test_preds = np.zeros(test_df.shape[0])

    kf = KFold(n_splits=5, random_state=2021, shuffle=True)

    for fold, (tr_idx, val_idx) in enumerate(kf.split(train_df)):
        print(f"Training fold {fold + 1}")

        X_tr, X_val = train_df.iloc[tr_idx][features], train_df.iloc[val_idx][features]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        # Weighting by inverse squared pressure (optional, kept from original code)
        tr_weight = 1.0 / np.square(y_tr)
        val_weight = 1.0 / np.square(y_val)

        dtrain = lgb.Dataset(X_tr, label=y_tr, weight=tr_weight)
        dval = lgb.Dataset(X_val, label=y_val, weight=val_weight)

        # LightGBM 4.x uses callbacks for early stopping
        callbacks = [lgb.early_stopping(stopping_rounds=50, verbose=False)]

        model = lgb.train(
            params=params,
            train_set=dtrain,
            num_boost_round=1000,
            valid_sets=[dtrain, dval],
            feval=feval_rmspe,
            callbacks=callbacks,
        )

        # Predict on validation and test sets
        oof_preds[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
        test_preds += (
            model.predict(test_df[features], num_iteration=model.best_iteration)
            / kf.n_splits
        )

    overall_rmspe = rmspe(y, oof_preds)
    print(f"Overall OOF RMSPE: {overall_rmspe:.5f}")
    return test_preds


params0 = {
    "objective": "mae",
    "boosting_type": "gbdt",
    "max_depth": -1,
    "max_bin": 100,
    "min_data_in_leaf": 500,
    "learning_rate": 0.05,
    "subsample": 0.72,
    "subsample_freq": 4,
    "feature_fraction": 0.5,
    "lambda_l1": 0.5,
    "lambda_l2": 1.0,
    # No categorical features in this dataset; keep placeholder for compatibility
    "categorical_column": [],
    "seed": 2021,
    "feature_fraction_seed": 2021,
    "bagging_seed": 2021,
    "drop_seed": 2021,
    "data_random_seed": 2021,
    "n_jobs": -1,
    "verbose": -1,
}



In [4]:
preds = train_and_evaluate_lgb(train, test, params0)

# Ensure pressures are non‑negative (physical constraint)
preds = np.clip(preds, a_min=0, a_max=None)

submission = test[["id"]].copy()
submission["pressure"] = preds
submission.to_csv("submission.csv", index=False)
print("Submission file 'submission.csv' created.")

Training fold 1
Training fold 2
Training fold 3
Training fold 4
Training fold 5
Overall OOF RMSPE: 2.59092
Submission file 'submission.csv' created.
